# LangGraph와 AgentCore Memory - Human-in-the-Loop(단기 메모리)

## 소개
이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LangGraph와 통합하여 **human-in-the-loop** workflow를 만드는 방법을 살펴봅니다. **단기 메모리** 유지와 사람의 개입을 위한 Agent 실행 중단 기능을 결합하여 자연스러운 인계가 가능한 정교한 고객 지원 시나리오를 구현하는 데 중점을 둡니다.

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화                                                        |
| Agent 사용 사례       | 사람에게 에스컬레이션하는 고객 지원                                          |
| Agentic Framework   | Langgraph                                                                        |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, Langgraph Checkpointer, Human-in-the-Loop        |
| 예제 난이도  | 초급                                                                     |

다음 내용을 학습합니다.
- workflow 유지를 위한 AgentCore Memory Checkpointer 생성
- human-in-the-loop workflow에 LangGraph의 interrupt 메커니즘 사용
- 사람의 개입을 위해 실행을 일시 중지할 수 있는 도구 구현
- 사람의 입력 후 LangGraph Command를 사용하여 Agent workflow 재개
- 자연스러운 인계가 필요한 복잡한 고객 지원 시나리오 관리

### 시나리오 배경

이 예제에서는 복잡한 문제를 Human Supervisor에게 에스컬레이션할 수 있는 "**Customer Support Agent**"를 만듭니다. 사람의 전문 지식이 필요한 상황이 발생하면 Agent는 실행을 일시 중지하고 현재 상태를 AgentCore Memory에 저장한 후 사람의 개입을 기다립니다. 이후 Human Supervisor가 지침을 제공하면 강화된 컨텍스트로 Agent 실행이 재개됩니다.

## 아키텍처
<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## 사전 요구 사항

- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory에 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델에 대한 액세스

### 통합 작동 방식

human-in-the-loop workflow에서 LangGraph와 AgentCore Memory의 통합은 다음과 같이 작동합니다.

1. 지속적인 상태 관리를 위한 Checkpointer backend로 AgentCore Memory 사용
2. 특정 시점에 실행을 일시 중지하는 interrupt 메커니즘 구현
3. Human Supervisor가 추가 컨텍스트로 workflow를 재개하도록 지원
4. 실행 중단 전후로 대화 기록과 상태 유지

이 접근 방식은 AI Agent와 Human Supervisor가 자연스럽게 협업하는 지원 workflow를 구현합니다.

먼저 환경을 설정하겠습니다.

In [ ]:
# 필요한 라이브러리 설치
!pip install -qr requirements.txt

In [ ]:
# LangGraph 및 LangChain 구성 요소 가져오기
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# human-in-the-loop 구현에 필요한 항목 가져오기
from langgraph.types import Command, interrupt

In [ ]:
import os
import logging

from bedrock_agentcore.memory import MemoryClient

# Checkpointer로 사용할 AgentCoreMemorySaver 가져오기
from langgraph_checkpoint_aws import AgentCoreMemorySaver

logging.getLogger("support-agent").setLevel(logging.INFO)
region = os.getenv("AWS_REGION", "us-west-2")

logger = logging.getLogger("support-agent")

## 1단계: Memory 생성
이 섹션에서는 AgentCore Memory SDK를 사용하여 Memory 저장소를 생성합니다. 이 Memory는 LangGraph Checkpointer의 backend 역할을 하며 지속적인 human-in-the-loop workflow를 지원합니다.

In [ ]:
memory_name = "SupportAgent"

client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory["id"]

### AgentCore Memory 구성

이제 AgentCore Memory Checkpointer를 구성하고 LLM을 초기화합니다.

- `memory_id`는 checkpoint가 저장될 AgentCore Memory 리소스에 해당합니다.
- `region`은 리소스의 AWS 리전을 지정합니다.
- `MODEL_ID`는 LangGraph Agent를 구동할 Bedrock 모델을 정의합니다.

`memory_id`와 추가 boto3 client keyword argument(이 예제에서는 `region`)를 사용하여 Checkpointer 인스턴스를 생성합니다.

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# 상태 유지를 위한 Checkpointer 초기화
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

## 2단계: Human-in-the-Loop Tool
Support Agent가 사용할 도구를 정의합니다. LangGraph의 `interrupt` type을 사용하면 Agent Graph 실행을 중단하여 사람이 개입하고 질의에 응답한 뒤 실행을 계속할 수 있습니다.


In [ ]:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]


@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b


@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply, human_assistance]

## 3단계: LangGraph Agent 구현

이제 AgentCore Memory Checkpointer와 human-in-the-loop 기능을 적용하여 LangGraph의 `create_react_agent` builder로 Support Agent를 생성합니다.

In [ ]:
# LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## 4단계: Support Agent 실행
이제 AgentCore Memory Checkpointer와 human-in-the-loop가 통합된 Agent를 실행할 수 있습니다. 이 예제에서는 사용자의 지원을 명시적으로 요청합니다. 실제 환경에서는 특정 키워드가 사용될 때 safety flag가 대화를 사람에게 전달하는 등 여러 조건으로 이 동작을 trigger할 수 있습니다.

### 구성 설정
LangGraph에서 config는 사용자 ID나 세션 ID처럼 호출 시 필요한 속성을 담는 `RuntimeConfig`입니다. 자세한 내용은 [추가 문서](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)에서 확인할 수 있습니다.

AgentCore Memory Checkpointer(`AgentCoreMemorySaver`)에는 다음 항목을 지정해야 합니다.
- `thread_id`: AgentCore session_id(고유 대화 thread)에 매핑
- `actor_id`: AgentCore actor_id(사용자, Agent 또는 기타 식별자)에 매핑

### Graph 호출 입력
가장 최근의 사용자 메시지만 `inputs` argument로 전달하면 됩니다. 다른 상태 변수를 포함할 수도 있지만, 단순한 `create_react_agent`에서는 메시지만 필요합니다.


In [ ]:
user_input = "I would like to work with a customer service human agent."
config = {"configurable": {"thread_id": "1", "actor_id": "demo-notebook"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

### Workflow 중단

Human Assistance Tool이 호출되었을 때 실행이 일시 중지되는 것을 확인할 수 있습니다. 현재 상태를 살펴보고 workflow가 중단된 지점을 확인해 보겠습니다.

In [ ]:
snapshot = graph.get_state(config)
snapshot.next

### Human Supervisor 개입

이제 Human Supervisor 역할로 지원 내용을 제공하고 LangGraph `Command`로 응답을 전송하여 workflow를 재개합니다. AgentCore Memory Checkpointer가 전체 대화 상태를 유지하므로 대화를 이어 갈 수 있습니다.

In [ ]:
human_response = "I'm sorry to hear that you are frustrated. Looking at the past conversation history, I can see that you've requested a refund. I've gone ahead and credited it to your account."

human_command = Command(resume={"messages": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

1. human-in-the-loop workflow용 AgentCore Memory 리소스를 생성하는 방법
2. interrupt 기능을 갖춘 LangGraph Agent 구축
3. 사람의 개입을 위해 실행을 일시 중지할 수 있는 도구 구현
4. 실행 중단 중 workflow 상태를 유지하기 위해 AgentCoreMemorySaver 사용
5. 사람이 제공한 컨텍스트로 Agent 실행 재개

이 통합은 LangGraph의 human-in-the-loop 기능과 AgentCore Memory의 강력한 상태 유지 기능을 결합하여 AI Agent와 Human Supervisor가 자연스럽게 협업하는 정교한 고객 지원 workflow를 만드는 방법을 보여 줍니다.

이 접근 방식은 다단계 에스컬레이션, 전문 인력 routing, 복잡한 승인 workflow 등 더 복잡한 시나리오로 확장할 수 있습니다.

## 리소스 정리
이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.


In [ ]:
# client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)